# 26. The stack, with XGBoost

**One variable against ledger row 34** (`stack_logit_24_oof`, CV 0.967807): the same
logistic combiner, the same `C`, the same fold-wise protocol, the same folds. The
member set goes from twenty-four to twenty-five, adding `xgb_te` from row 38.

Row 34 is **refit in this run rather than quoted**, so the comparison is paired inside
one notebook, and its refit has a number it is required to reproduce.

## This is the gate that was pre-registered in `24_xgboost_te.ipynb`

Written there before XGBoost had been run, and repeated here unchanged:

> XGBoost joins if its leave-one-out contribution to a fold-wise 25-member stack is
> **positive on at least 4 of the 5 folds and at least +0.00005 on the mean**, which
> is the floor `17_catboost_te.ipynb` used for the same decision about CatBoost.

The floor is not arbitrary. It is what `17` required of CatBoost, and CatBoost went on
to be worth +0.000100 as a nineteenth member (row 27).

## Why the single-model CV does not decide this

Row 38 made XGBoost the best single model in the repo at 0.967099, above CatBoost's
0.966915 and LightGBM's 0.966782. That is not the question. Two results in this repo
say a fitted combiner's verdict does not follow from a member's strength:

- **Row 16 and row 34.** The neural model was 0.0247 behind and still earned +0.1178,
  because a combiner can use a weak member as a correction rather than averaging it in.
- **Row 34's ratio.** Improving that same neural model by 0.026 bought **+0.000043**
  in the stack, because the weak version was already supplying most of that direction.

So a strong new member can be worth little if the stack already has its direction
covered, and that is the specific risk here: `xgb_te` is a third histogram GBDT sitting
beside five LightGBM seeds and five CatBoost seeds on the identical feature set.

## What to read

The CV step is expected to be small. **The coefficients are the result**, as they were
in rows 27, 32 and 34: whether the combiner gives `xgb_te` its own weight or takes it
out of the GBDTs already present says which of the two stories above is true.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def find_repo():
    for b in [Path.cwd(), *Path.cwd().parents]:
        if (b / "data" / "raw" / "train.csv").exists():
            return b
    raise FileNotFoundError("data/raw/train.csv not found")


REPO = find_repo()
O, S = REPO / "artifacts" / "oof", REPO / "submissions"

train = pd.read_csv(REPO / "data" / "raw" / "train.csv")
test = pd.read_csv(REPO / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy()

# The same split every vector on disk was produced under. Rebuilt rather than loaded,
# and then checked, because a silently different fold vector is the one error here
# that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]


In [2]:
# Row 32's twenty-three in row 32's order, then the target-encoded neural model.
MEM = [
    ("te42", O / "te_bag42_oof.npy", O / "te_bag42_test.npy"),
    ("te2024", O / "te_seed2024_oof.npy", O / "te_seed2024_test.npy"),
    ("te7", O / "te_seed7_oof.npy", O / "te_seed7_test.npy"),
    ("te2025", O / "te_seed2025_oof.npy", O / "te_seed2025_test.npy"),
    ("te13", O / "te_seed13_oof.npy", O / "te_seed13_test.npy"),
    ("anchor", O / "lgbm_default_anchor_seed42.npy",
     S / "lgbm_default_anchor_seed42.csv"),
    ("trees300", O / "lgbm_trees300_seed42.npy", S / "lgbm_trees300_seed42.csv"),
    ("trees1000", O / "lgbm_trees1000_seed42.npy", S / "lgbm_trees1000_seed42.csv"),
    ("trees2000", O / "lgbm_trees2000_seed42.npy", S / "lgbm_trees2000_seed42.csv"),
    ("lr010", O / "lgbm_lr01_n1000_seed42.npy", S / "lgbm_lr01_n1000_seed42.csv"),
    ("lr005", O / "lgbm_lr005_n2000_seed42.npy", S / "lgbm_lr005_n2000_seed42.csv"),
    ("lr003", O / "lgbm_lr003_n3333_seed42.npy", S / "lgbm_lr003_n3333_seed42.csv"),
    ("bag42", O / "lgbm_bag08_lr005_n2000_seed42.npy",
     S / "lgbm_bag08_lr005_n2000_seed42.csv"),
    ("bag2024", O / "lgbm_bag08_lr005_n2000_seed2024.npy",
     S / "lgbm_bag08_lr005_n2000_seed2024.csv"),
    ("bag7", O / "lgbm_bag08_lr005_n2000_seed7.npy",
     S / "lgbm_bag08_lr005_n2000_seed7.csv"),
    ("bag2025", O / "lgbm_bag08_lr005_n2000_seed2025.npy",
     S / "lgbm_bag08_lr005_n2000_seed2025.csv"),
    ("bag13", O / "lgbm_bag08_lr005_n2000_seed13.npy",
     S / "lgbm_bag08_lr005_n2000_seed13.csv"),
    ("neural", O / "neural_oof.npy", O / "neural_test.npy"),
    ("cat42", O / "catboost_te_oof.npy", O / "catboost_te_test.npy"),
    ("cat2024", O / "catboost_te_seed2024_oof.npy",
     O / "catboost_te_seed2024_test.npy"),
    ("cat7", O / "catboost_te_seed7_oof.npy", O / "catboost_te_seed7_test.npy"),
    ("cat2025", O / "catboost_te_seed2025_oof.npy",
     O / "catboost_te_seed2025_test.npy"),
    ("cat13", O / "catboost_te_seed13_oof.npy", O / "catboost_te_seed13_test.npy"),
    ("neural_te", O / "neural_te_oof.npy", O / "neural_te_test.npy"),
    ("xgb_te", O / "xgb_te_oof.npy", O / "xgb_te_test.npy"),
]
NEW = ["xgb_te"]
CATS = ["cat42", "cat2024", "cat7", "cat2025", "cat13"]


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


def load_test(path):
    if path.suffix == ".npy":
        return np.load(path)
    df = pd.read_csv(path)
    # A csv written in a different row order would blend perfectly cleanly and be
    # undetectable in the score. Checked rather than assumed.
    assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {path.name}"
    return df["addicted_label"].to_numpy()


names = [m[0] for m in MEM]
Poof = {n: np.load(p) for n, p, _ in MEM}
Ptest = {n: load_test(t) for n, _, t in MEM}

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
KEEP24 = [i for i, n in enumerate(names) if n not in NEW]
print(f"{len(names)} members, oof {Loof.shape}, test {Ltest.shape}")
print("member CV:")
for n in names:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    print(f"  {n:10} {cv:.6f}" + ("   <- new" if n in NEW else ""))

25 members, oof (691369, 25), test (296302, 25)
member CV:


  te42       0.966782


  te2024     0.966771


  te7        0.966729


  te2025     0.966743


...

  cat7       0.966920


  cat2025    0.966916


  cat13      0.966922


  neural_te  0.965373


  xgb_te     0.967099   <- new


In [3]:
# The fold loop, run twice over the identical folds: once on row 34's twenty-four and
# once with XGBoost added. Identical to 22's, and to row 25's before it.
def run(cols):
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)],
                                                           y[tr])
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf


per24, test24, coef24 = run(KEEP24)
per25, test25, coef25 = run(list(range(len(names))))

ROW34_CV = 0.967807
repro = per24.mean() - ROW34_CV
REPRODUCED = abs(repro) < 1e-4

print(f"{'':22} {'fold 0':>9} {'fold 1':>9} {'fold 2':>9} {'fold 3':>9} {'fold 4':>9}")
for lbl, p in (("24 members, row 34", per24), ("25 members, this run", per25)):
    print(f"{lbl:22} " + " ".join(f"{v:9.6f}" for v in p))
print()
print(f"24-member CV {per24.mean():.6f} +/- {per24.std():.6f}"
      f"   (row 34 recorded {ROW34_CV:.6f}, diff {repro:+.2e})")
print(f"25-member CV {per25.mean():.6f} +/- {per25.std():.6f}")
if not REPRODUCED:
    print("\nROW 34 DID NOT REPRODUCE. Nothing below is comparable to it.")

                          fold 0    fold 1    fold 2    fold 3    fold 4
24 members, row 34      0.967159  0.967964  0.968124  0.968325  0.967462
25 members, this run    0.967251  0.967991  0.968175  0.968413  0.967534

24-member CV 0.967807 +/- 0.000432   (row 34 recorded 0.967807, diff -4.34e-07)
25-member CV 0.967873 +/- 0.000424


In [4]:
# Paired, on identical folds. The right test when two models share folds is the spread
# of the per-fold DIFFERENCES and how many folds it wins, not the fold spread, which is
# common to both and cancels. See NOTES.md.
def paired(a, b, lbl):
    d = a - b
    t = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    print(f"{lbl:38} {d.mean():+.6f}  sd {d.std(ddof=1):.6f}  "
          f"{(d > 0).sum()}/5  t(4)={t:.2f}")
    print("     per fold: " + "  ".join(f"{v:+.6f}" for v in d))
    return d


base17 = np.array([roc_auc_score(y[folds == f], Poof["te42"][folds == f])
                   for f in range(5)])
base38 = np.array([roc_auc_score(y[folds == f], Poof["xgb_te"][folds == f])
                   for f in range(5)])
d_new = paired(per25, per24, "25 vs 24 members (row 34)")
paired(per25, base17, "25-member vs te42 alone (row 17)")
paired(per25, base38, "25-member vs xgb_te alone (row 38)")
print()
print("For scale, from this repo's own stack history:")
print("  18 -> 19, adding CatBoost           +0.000100  (row 27)")
print("  19 -> 23, four more CatBoost seeds  +0.000014  (row 32)")
print("  23 -> 24, adding neural_te          +0.000043  (row 34)")

25 vs 24 members (row 34)              +0.000066  sd 0.000027  5/5  t(4)=5.50
     per fold: +0.000092  +0.000028  +0.000051  +0.000088  +0.000072
25-member vs te42 alone (row 17)       +0.001090  sd 0.000075  5/5  t(4)=32.62
     per fold: +0.001141  +0.001016  +0.001002  +0.001149  +0.001144
25-member vs xgb_te alone (row 38)     +0.000774  sd 0.000067  5/5  t(4)=25.94
     per fold: +0.000696  +0.000850  +0.000815  +0.000713  +0.000796

For scale, from this repo's own stack history:
  18 -> 19, adding CatBoost           +0.000100  (row 27)
  19 -> 23, four more CatBoost seeds  +0.000014  (row 32)
  23 -> 24, adding neural_te          +0.000043  (row 34)


In [5]:
# The gate, exactly as pre-registered in 24_xgboost_te.ipynb before XGBoost was run.
GATE_FLOOR = 5e-05
wins = int((d_new > 0).sum())
mean_g = float(d_new.mean())

print(f"gate: mean {mean_g:+.6f} against a floor of {GATE_FLOOR:+.6f}, "
      f"wins {wins}/5")
print()
if not REPRODUCED:
    print("VERDICT: blocked, row 34 did not reproduce in this run")
elif wins >= 4 and mean_g >= GATE_FLOOR:
    print("VERDICT: XGBoost joins the stack. The gate was met on both conditions.")
elif wins >= 4 and mean_g > 0:
    print("VERDICT: positive but under the pre-registered floor. Per NOTES.md this is")
    print("  inconclusive, not an improvement, and the floor was set before the run")
    print("  precisely so it could not be moved now.")
else:
    print("VERDICT: XGBoost does not earn a place. The best single model in the repo")
    print("  adds nothing a fitted combiner wants, which is the row 34 ratio again in")
    print("  a sharper form.")

gate: mean +0.000066 against a floor of +0.000050, wins 5/5

VERDICT: XGBoost joins the stack. The gate was met on both conditions.


In [6]:
# Coefficients. The question the header set up: does xgb_te earn its own weight, or
# take it out of the ten GBDT members already sitting on this feature set?
# Row 34's coefficients are the in-run refit rather than a hardcoded table, so the
# comparison cannot drift from what row 34 actually was.
was = {names[i]: coef24[:, i].mean() for i in range(len(KEEP24))}

print(f"{'member':10} {'25-member':>11} {'fold sd':>9}   {'row 34':>9}  {'change':>8}")
for i in np.argsort(-coef25.mean(axis=0)):
    n = names[i]
    now = coef25[:, i].mean()
    if n in NEW:
        tail = f"{'new':>9}  {'':>8}"
    else:
        tail = f"{was[n]:>+9.4f}  {now - was[n]:>+8.4f}"
    print(f"{n:10} {now:>+11.4f} {coef25[:, i].std():>9.4f}   {tail}")

GBDT = [n for n in names if n.startswith(("te", "cat", "bag", "trees", "lr"))
        and n != "xgb_te"]
g_now = sum(coef25[:, names.index(n)].mean() for n in GBDT)
g_was = sum(was[n] for n in GBDT)
print(f"\nxgb_te coefficient                 : "
      f"{coef25[:, names.index('xgb_te')].mean():+.4f}")
print(f"sum of the other {len(GBDT)} GBDT coefficients: {g_now:+.4f}   "
      f"(row 34: {g_was:+.4f}, change {g_now - g_was:+.4f})")
print(f"largest fold-to-fold sd            : {coef25.std(axis=0).max():.4f}")
print()
print("If xgb_te's weight is roughly matched by a fall in the other GBDTs, the")
print("combiner is substituting rather than adding, which is what row 32 found when")
print("four CatBoost seeds split one weight between them.")

member       25-member   fold sd      row 34    change
xgb_te         +0.2751    0.0137         new          
neural_te      +0.1316    0.0042     +0.1305   +0.0012
lr003          +0.1084    0.0153     +0.1166   -0.0082
lr005          +0.0906    0.0095     +0.0954   -0.0049
neural         +0.0842    0.0022     +0.0905   -0.0064
bag42          +0.0784    0.0116     +0.0871   -0.0087
bag7           +0.0766    0.0170     +0.0826   -0.0059
bag13          +0.0741    0.0072     +0.0796   -0.0055
cat2024        +0.0699    0.0098     +0.0799   -0.0101
cat13          +0.0573    0.0078     +0.0657   -0.0084
cat2025        +0.0559    0.0039     +0.0650   -0.0092
bag2025        +0.0519    0.0059     +0.0573   -0.0054
cat42          +0.0503    0.0038     +0.0583   -0.0080
cat7           +0.0439    0.0072     +0.0524   -0.0084
te13           +0.0389    0.0067     +0.0842   -0.0452
te2024         +0.0358    0.0128     +0.0825   -0.0467
te2025         +0.0358    0.0129     +0.0819   -0.0461
te42      

In [7]:
# Submission file, matching what every other stack row in this repo writes. Writing it
# is not submitting it: the two remaining slots are a separate decision and this
# notebook does not spend one.
pred = test25.mean(axis=0)
prob = 1 / (1 + np.exp(-pred))

prev = pd.read_csv(S / "stack_oof_24.csv")
assert (prev["id"].to_numpy() == test["id"].to_numpy()).all()
t34 = pd.Series(pred).corr(pd.Series(prev["addicted_label"].to_numpy()),
                           method="spearman")
t34b = pd.Series(pred).corr(pd.Series(test24.mean(axis=0)), method="spearman")
print(f"spearman vs row 34 on disk    : {t34:.7f}")
print(f"spearman vs row 34 refit here : {t34b:.7f}")
print("Row 25 sat at 0.9999995 against row 24 and was held back on that basis, so a")
print("number that close means the leaderboard cannot tell these two apart.")

out = S / "stack_oof_25.csv"
sub = pd.DataFrame({"id": test["id"], "addicted_label": prob})
assert len(sub) == len(test) and sub["addicted_label"].between(0, 1).all()
sub.to_csv(out, index=False)
print(f"\nwrote {out.name}, {len(sub):,} rows, "
      f"range [{prob.min():.4f}, {prob.max():.4f}]")
print(f"ledger: CV {per25.mean():.6f} +/- {per25.std():.6f}, "
      f"vs row 34 {d_new.mean():+.6f} ({wins}/5, sd {d_new.std(ddof=1):.6f})")

spearman vs row 34 on disk    : 0.9998222
spearman vs row 34 refit here : 0.9998222
Row 25 sat at 0.9999995 against row 24 and was held back on that basis, so a
number that close means the leaderboard cannot tell these two apart.



wrote stack_oof_25.csv, 296,302 rows, range [0.0000, 1.0000]
ledger: CV 0.967873 +/- 0.000424, vs row 34 +0.000066 (5/5, sd 0.000027)
